# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, their IDs, and columns.

We will print the available record sets and their field/column IDs. All references use the `@id` field as required.

In [ ]:
# List record sets and fields by @id
print("Available record sets (by @id):\n")
record_sets = []
for rs in dataset.metadata.record_sets:
    print(f"Record Set Name: {rs.name} | @id: {rs.id}")
    record_sets.append(rs.id)
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} | @id: {field.id}")
        if hasattr(field, 'column') and field.column is not None:
            print(f"      Column @id: {field.column.id if hasattr(field.column,'id') else field.column}")
    print("  Columns:")
    if rs.columns:
        for col in rs.columns:
            print(f"    - {col.name} | @id: {col.id}")
    print('')
if record_sets:
    print(f"Total record sets found: {len(record_sets)}")
else:
    print("No record sets found in metadata.")

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. Use the record set and field `@id`s observed above.

**Note:** You must supply the correct `record_set_id` (from overview above) to extract records from the dataset.

In [ ]:
# Example: Extract data from each record set
import mlcroissant as mlc
import pandas as pd

# Record sets to extract data from (supply as a list of @id)
record_sets_to_load = record_sets  # from previous cell; or set manually, e.g. ['cr:recordSet_1']
dataframes = {}

for record_set_id in record_sets_to_load:
    print(f"Loading records from Record Set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(df.columns.tolist())
        print(df.head())
    else:
        print(f"No records found for Record Set {record_set_id}.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalization, and grouping on one of the loaded record sets and fields.

All fields and record set references use the `@id` for clarity and reproducibility.

In [ ]:
# Choose a record set and numeric field by their @id

# EXAMPLE Placeholders:
# record_set_id = 'cr:recordSet_PROTEINS'
# numeric_field_id = 'cr:field_MolecularWeight'
# group_field_id = 'cr:field_SampleType'

# Use actual IDs from overview above
# If unsure, use cell [2] to print actual available IDs

if dataframes:
    # For demo, pick the first loaded record set
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Analyzing record set: {record_set_id}")

    # Try to automatically detect a numeric field (float or int column)
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if numeric_field_id is None:
        print("No numeric field found in DataFrame columns for further EDA.")
    else:
        print(f"Using numeric field: {numeric_field_id}")
        # Filtering: Example threshold (e.g. > 10)
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalization (Z-score)
        normalized_col = numeric_field_id + '_normalized'
        filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, normalized_col]].head())

        # Try to group by a likely categorical field (non-numeric column)
        group_field_id = None
        for col in df.columns:
            if not pd.api.types.is_numeric_dtype(df[col]):
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
            print(grouped_df.head())
        else:
            print("No suitable categorical column found for grouping.")
else:
    print("No DataFrames found for EDA. Please check previous step.")

## 5. Visualization
Visualize distributions or relationships between fields in the dataset. The following example creates a histogram and a boxplot for the selected numeric field.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id is not None and record_set_id in dataframes:
    plt.figure(figsize=(10, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f'Histogram of {numeric_field_id} in Record Set {record_set_id}')
    plt.xlabel(numeric_field_id)
    plt.show()
    
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=df[numeric_field_id].dropna())
    plt.title(f'Boxplot of {numeric_field_id} in Record Set {record_set_id}')
    plt.xlabel(numeric_field_id)
    plt.show()
else:
    print("Cannot plot as no numeric field/data is available.")

## 6. Conclusion
This notebook demonstrated how to use the `mlcroissant` library to load, explore, and process a FAIR Croissant-formatted dataset using only `@id` references for record sets and fields. You can further extend the EDA and visualizations to suit your research needs. 

**Key findings and observations:**
- The dataset provides experimental protein abundance and modification data from human extracellular vesicle samples.
- With the structured Croissant schema, one can easily identify available fields, types, and reference them precisely using their `@id`.
- Filtering and normalization can be applied to numeric protein data, enabling downstream biomarker or comparative analysis.
